# Back-Translation with Qwen (English → Chinese → English)

Takes a file of English texts and, for each text, makes two successive calls
to Qwen: the text is translated into Standard (Simplified) Chinese, then the
Chinese translation is translated back into English. Writes a JSON file
containing the back-translated English texts only.

Each output stays within a fixed range of the input length (in LLaDA tokens)
and never exceeds the upper bound.

### Structure

1. Setup
2. Configuration
3. Input texts
4. Length measurement
5. Qwen
6. Prompts and generation functions
7. Generation (with resume)
8. Output file

## 1. Setup

**Runtime:** this notebook needs a **GPU** (Runtime → Change runtime type → GPU).
A free T4 works with 4-bit quantization; an L4 or A100 is faster.

Put your input file in Google Drive and set `INPUT_FILE` below.

In [1]:
%pip install -q bitsandbytes accelerate

In [2]:
import json
import math
import re
from collections import Counter
from pathlib import Path
from statistics import median

import torch
from google.colab import drive

drive.mount("/content/drive")
assert torch.cuda.is_available(), "No GPU found: change the runtime type to GPU."
print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
GPU: NVIDIA L4


## 2. Configuration

`N_TEXTS = 12` processes only 12 texts, evenly spread over the file, so
that the three datasets are represented (useful for a test). `None`
processes all of them, and output line *i* then matches input line *i*.

In [3]:
# --- Files -----------------------------------------------------------------
INPUT_FILE = Path("/content/drive/MyDrive/watermark_project/watermarked_600.jsonl")
OUTPUT_FILE = Path("/content/drive/MyDrive/watermark_project/backtranslated_texts.json")
TEXT_FIELD = "completion"   # the watermarked answer (not "prompt", which is the question)
N_TEXTS = None              # None = all texts; e.g. 12 for a test

# --- Qwen ------------------------------------------------------------------
MODEL = "Qwen/Qwen2.5-7B-Instruct"
SEED = 0
TEMPERATURE = 0.7
TOP_P = 0.9
BATCH_SIZE = 8         # lower it if the GPU runs out of memory

# --- Length constraint (LLaDA tokens) --------------------------------------
MIN_LENGTH_RATIO = 0.80
MAX_LENGTH_RATIO = 1.10   # never exceeded in the output
MAX_RETRIES = 3           # regenerations for an invalid output

# --- Language checks -------------------------------------------------------
MIN_CHINESE_SHARE = 0.50  # step 1: share of Chinese characters among letters, at least
MAX_CHINESE_SHARE = 0.02  # step 2: share of Chinese characters among letters, at most

# Progress file (used to resume after a disconnection); one per output file and size
suffix = "" if N_TEXTS is None else f"_n{N_TEXTS}"
PROGRESS_FILE = OUTPUT_FILE.with_name(OUTPUT_FILE.stem + suffix + "_progress.jsonl")
OUTPUT_FILE = OUTPUT_FILE.with_name(OUTPUT_FILE.stem + suffix + OUTPUT_FILE.suffix)
print("Output:", OUTPUT_FILE)

Output: /content/drive/MyDrive/watermark_project/backtranslated_texts.json


## 3. Input Texts

Accepted input formats:

- a `.json` file containing a list of texts: `["text 1", "text 2", ...]`;
- a `.json` file containing a list of objects, e.g. `[{"text": "..."}, ...]`;
- a `.jsonl` file with one text (or one object) per line.

For objects, the text is read from the field `TEXT_FIELD`. If it is `None`,
the first of `text`, `completion`, `content` found is used.

In [4]:
def read_texts(path, field=None):
    # Read a list of texts from a .json or .jsonl file
    path = Path(path)
    if path.suffix == ".jsonl":
        items = [json.loads(line) for line in path.open(encoding="utf-8") if line.strip()]
    else:
        items = json.load(path.open(encoding="utf-8"))
    texts = []
    for k, item in enumerate(items):
        if isinstance(item, str):
            texts.append(item)
            continue
        keys = [field] if field else ["text", "completion", "content"]
        key = next((key for key in keys if key in item), None)
        assert key is not None, f"Item {k}: no text field among {keys} (fields: {list(item)})"
        texts.append(item[key])
    return texts

texts = read_texts(INPUT_FILE, TEXT_FIELD)
source_indices = list(range(len(texts)))       # line of each text in the input file
if N_TEXTS is not None:
    assert 0 < N_TEXTS <= len(texts), f"N_TEXTS must be between 1 and {len(texts)}"
    step = len(texts) / N_TEXTS
    source_indices = [int(k * step) for k in range(N_TEXTS)]   # evenly spread over the file
    texts = [texts[i] for i in source_indices]
print("Texts to process:", len(texts))

Texts to process: 600


## 4. Length Measurement

Lengths are measured with the **LLaDA tokenizer**, the tokenizer of the
watermarked model, so that the length limit is expressed in the same tokens
as the ones the watermark is computed on.

Each output must satisfy:

    MIN_LENGTH_RATIO × input length  ≤  output length  ≤  MAX_LENGTH_RATIO × input length

In [5]:
from transformers import AutoTokenizer

llada_tok = AutoTokenizer.from_pretrained("GSAI-ML/LLaDA-8B-Instruct", trust_remote_code=True)

def n_llada(text):
    # Number of LLaDA tokens
    return len(llada_tok(text, add_special_tokens=False)["input_ids"])

def length_bounds(text):
    # Allowed range [low, high] for the output, in LLaDA tokens
    n = n_llada(text)
    return math.ceil(MIN_LENGTH_RATIO * n), math.floor(MAX_LENGTH_RATIO * n)

lengths = [n_llada(t) for t in texts]
print(f"Input lengths (LLaDA tokens): min {min(lengths)}, median {median(lengths):.0f}, max {max(lengths)}")

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

configuration_llada.py:   0%|          | 0.00/12.0k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/GSAI-ML/LLaDA-8B-Instruct:
- configuration_llada.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/51.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/6.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

Input lengths (LLaDA tokens): min 42, median 298, max 299


## 5. Qwen

The model is loaded in 4-bit on GPUs with less than 20 GB of memory (e.g. T4),
and in 16-bit otherwise.

In [6]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

qtok = AutoTokenizer.from_pretrained(MODEL)
qtok.padding_side = "left"            # required for batched generation with decoder-only models
if qtok.pad_token is None:
    qtok.pad_token = qtok.eos_token

gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
USE_4BIT = gpu_gb < 20

if USE_4BIT:
    qwen = AutoModelForCausalLM.from_pretrained(
        MODEL, device_map="auto",
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=dtype),
    )
else:
    qwen = AutoModelForCausalLM.from_pretrained(MODEL, device_map="auto", torch_dtype=dtype)
qwen.eval()
torch.manual_seed(SEED)

# Token ids that mark a normal end of the answer
eos = qwen.generation_config.eos_token_id
STOP_IDS = set(eos if isinstance(eos, list) else [eos])

print(f"Loaded {MODEL} | GPU memory {gpu_gb:.0f} GB | 4-bit: {USE_4BIT} | dtype: {dtype}")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-7B-Instruct | GPU memory 24 GB | 4-bit: False | dtype: torch.bfloat16


## 6. Prompts and Generation Functions

- **Prompts.** Step 1 asks for a faithful translation into Simplified Chinese.
  Step 2 asks for a faithful translation into English, of about the original
  number of words and never longer. In both steps the input is placed between
  `<<<` and `>>>`, and Qwen must write only its translation, starting with
  `<<<` and ending with `>>>`.
- **Extraction.** The translation is the text between the first `<<<` and the
  last `>>>`; anything around them is ignored. An answer with a missing
  delimiter, or with several blocks, is invalid and regenerated.
- **Language checks.** After step 1, at least `MIN_CHINESE_SHARE` of the
  letters must be Chinese characters; after step 2, at most `MAX_CHINESE_SHARE`.
- **Retries.** Only the step that failed is redone (up to `MAX_RETRIES`
  times); a valid Chinese translation is kept. In the last round, step 2 is
  also run on Chinese translations that failed the check, so that every text
  gets an output.
- **Length.** The delimiters are removed before the length is measured. An
  English output still too long after all retries is cut at the last full
  sentence that fits.

In [7]:
def chat(content):
    messages = [{"role": "user", "content": content}]
    # enable_thinking=False turns off the reasoning mode of Qwen3 models; it is ignored by Qwen2.5
    return qtok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)

OPEN, CLOSE = "<<<", ">>>"   # delimiters of the text, in the prompt and in the answer

def extract(output):
    # Return (text between <<< and >>>, True if the answer has the expected format)
    output = re.sub(r"<think>.*?</think>", "", output, flags=re.S)   # Qwen3 reasoning, if any
    start = output.find(OPEN)
    if start == -1:                          # no opening delimiter
        return output.strip(), False
    rest = output[start + len(OPEN):]
    end = rest.rfind(CLOSE)                  # last >>>, in case the text itself contains >>>
    if end == -1:                            # opening only (answer cut by the length cap)
        return rest.strip(), False
    text = rest[:end].strip()
    if OPEN in text:                         # several blocks (e.g. two versions): not usable
        return text, False
    return text, True

@torch.no_grad()
def generate(prompts, max_new_tokens):
    # One answer per prompt; returns (extracted text, finished) pairs,
    # where finished = the model stopped by itself and wrote its text between <<< and >>>
    enc = qtok(prompts, return_tensors="pt", padding=True).to(qwen.device)
    gen = qwen.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True,
                        temperature=TEMPERATURE, top_p=TOP_P, pad_token_id=qtok.pad_token_id)
    results = []
    for ids in gen[:, enc["input_ids"].shape[1]:].tolist():
        text, well_formed = extract(qtok.decode(ids, skip_special_tokens=True))
        results.append((text, any(i in STOP_IDS for i in ids) and well_formed))
    return results

def n_qwen(text):
    return len(qtok(text)["input_ids"])

# End of a sentence: . ! or ?, optionally followed by a closing quote or bracket
SENTENCE_END = re.compile(r"[.!?][\"')\]]?(?=\s|$)")

def trim_to_tokens(text, max_tokens):
    # Longest prefix ending at a sentence boundary with at most max_tokens LLaDA tokens
    best = ""
    for m in SENTENCE_END.finditer(text):
        if n_llada(text[:m.end()]) <= max_tokens:
            best = text[:m.end()]
        else:
            break
    if best:
        return best.strip()
    # No full sentence fits: hard cut at max_tokens
    ids = llada_tok(text, add_special_tokens=False)["input_ids"][:max_tokens]
    cut = llada_tok.decode(ids, skip_special_tokens=True).strip()
    # Re-tokenizing the cut text can give one more token: remove tokens until it fits
    while ids and n_llada(cut) > max_tokens:
        ids = ids[:-1]
        cut = llada_tok.decode(ids, skip_special_tokens=True).strip()
    return cut

def violation(n, low, high):
    # Distance (in tokens) between a length and the allowed range; 0 = inside
    return max(low - n, n - high, 0)

def choose(attempts, low, high):
    # Keep the first valid attempt; otherwise the closest one, trimmed if too long or incomplete
    valid = [a for a in attempts if a["finished"] and a["text"] and low <= a["n"] <= high]
    if valid:
        return valid[0]["text"], "ok"
    chosen = min(attempts, key=lambda a: (violation(a["n"], low, high), not a["finished"]))
    if chosen["n"] > high or not chosen["finished"]:
        return trim_to_tokens(chosen["text"], high), "trimmed"
    return chosen["text"], "too_short"


TO_CHINESE = (
    "Translate the English text between <<< and >>> into Standard Chinese "
    "(Simplified characters). Translate all the information faithfully.\n"
    "Write only the translation, starting with <<< and ending with >>>.\n\n"
    "Text:\n"
    "<<<\n{text}\n>>>"
)

TO_ENGLISH = (
    "Translate the Chinese text between <<< and >>> into English. "
    "Translate all the information faithfully. "
    "Your translation must be about {n_words} words long and must not be longer.\n"
    "Write only the translation, starting with <<< and ending with >>>.\n\n"
    "Text:\n"
    "<<<\n{text}\n>>>"
)

CJK = re.compile(r"[㐀-䶿一-鿿豈-﫿]")

def chinese_share(text):
    # Share of Chinese characters among letters (spaces, digits and punctuation ignored)
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    return sum(bool(CJK.match(c)) for c in letters) / len(letters)

def process_batch(batch_texts):
    # Back-translate a batch of texts; returns one (output, status) pair per text
    bounds = [length_bounds(t) for t in batch_texts]
    chinese = [""] * len(batch_texts)            # latest Chinese translation of each text
    chinese_ok = [False] * len(batch_texts)
    attempts = [[] for _ in batch_texts]         # English attempts
    pending = list(range(len(batch_texts)))      # texts without a valid English output yet

    for round_ in range(1 + MAX_RETRIES):
        if not pending:
            break
        last_round = round_ == MAX_RETRIES

        # Step 1: English -> Chinese, where no valid translation exists yet
        need_zh = [i for i in pending if not chinese_ok[i]]
        if need_zh:
            prompts = [chat(TO_CHINESE.format(text=batch_texts[i])) for i in need_zh]
            cap = max(math.ceil(1.5 * n_qwen(batch_texts[i])) for i in need_zh) + 32
            for i, (text, finished) in zip(need_zh, generate(prompts, cap)):
                chinese[i] = text
                chinese_ok[i] = finished and chinese_share(text) >= MIN_CHINESE_SHARE

        # Step 2: Chinese -> English (also from invalid Chinese in the last round)
        need_en = [i for i in pending if chinese_ok[i] or last_round]
        if not need_en:
            continue
        prompts = [chat(TO_ENGLISH.format(n_words=len(batch_texts[i].split()), text=chinese[i]))
                   for i in need_en]
        # Cap in Qwen tokens: largest allowed output, plus room for <<< >>> and the end marker
        cap = max(math.ceil(MAX_LENGTH_RATIO * n_qwen(batch_texts[i])) for i in need_en) + 16
        for i, (text, finished) in zip(need_en, generate(prompts, cap)):
            n = n_llada(text)
            english = chinese_share(text) <= MAX_CHINESE_SHARE
            attempts[i].append({"text": text, "finished": finished and english, "n": n})
            low, high = bounds[i]
            if finished and english and text and low <= n <= high:
                pending.remove(i)

    results = []
    for k, (att, (low, high)) in enumerate(zip(attempts, bounds)):
        text, status = choose(att, low, high)
        if not chinese_ok[k]:
            status = "no_chinese"     # step 1 never gave Chinese: not a real back-translation
        elif chinese_share(text) > MAX_CHINESE_SHARE:
            status = "not_english"    # the kept output still contains Chinese
        results.append((text, status))
    return results

## 7. Generation

Each text needs two calls to Qwen, so this takes about twice as long as
paraphrasing. Progress is saved to Google Drive after every batch. If Colab
disconnects, run the notebook again: texts already done are skipped.

In [8]:
def read_progress():
    # Results already saved, by text index; a line cut by a disconnection is ignored
    done = {}
    if PROGRESS_FILE.exists():
        for line in PROGRESS_FILE.open(encoding="utf-8"):
            if not line.strip():
                continue
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue                       # incomplete line: that text will be redone
            done[r["index"]] = r
    return done

def run(texts):
    # Process all texts, saving progress after every batch; texts already done are skipped
    done = read_progress()
    # If the last line was cut, start a new line so that new results are not glued to it
    if PROGRESS_FILE.exists() and not PROGRESS_FILE.read_text(encoding="utf-8").endswith("\n"):
        with PROGRESS_FILE.open("a", encoding="utf-8") as f:
            f.write("\n")
    todo = [i for i in range(len(texts)) if i not in done]
    todo.sort(key=lambda i: len(texts[i]))     # similar lengths together: less padding
    print(f"Already done {len(done)} | remaining {len(todo)}")

    for start in range(0, len(todo), BATCH_SIZE):
        batch = todo[start:start + BATCH_SIZE]
        outputs = process_batch([texts[i] for i in batch])   # generate first, then write
        with PROGRESS_FILE.open("a", encoding="utf-8") as f:
            for i, (text, status) in zip(batch, outputs):
                f.write(json.dumps({"index": i, "text": text, "status": status},
                                   ensure_ascii=False) + "\n")
        print(f"  {min(start + BATCH_SIZE, len(todo))}/{len(todo)}")

run(texts)

Already done 0 | remaining 600
  8/600
  16/600
  24/600
  32/600
  40/600
  48/600
  56/600
  64/600
  72/600
  80/600
  88/600
  96/600
  104/600
  112/600
  120/600
  128/600
  136/600
  144/600
  152/600
  160/600
  168/600
  176/600
  184/600
  192/600
  200/600
  208/600
  216/600
  224/600
  232/600
  240/600
  248/600
  256/600
  264/600
  272/600
  280/600
  288/600
  296/600
  304/600
  312/600
  320/600
  328/600
  336/600
  344/600
  352/600
  360/600
  368/600
  376/600
  384/600
  392/600
  400/600
  408/600
  416/600
  424/600
  432/600
  440/600
  448/600
  456/600
  464/600
  472/600
  480/600
  488/600
  496/600
  504/600
  512/600
  520/600
  528/600
  536/600
  544/600
  552/600
  560/600
  568/600
  576/600
  584/600
  592/600
  600/600


## 8. Output File

## 8. Output File

The output is a JSON list of texts, **in the same order as the input**:
`["output for text 1", "output for text 2", ...]`. It contains the texts only.

The status of each text is printed below (and kept in the progress file):

- `ok`: valid back-translation within the length range;
- `trimmed`: too long or incomplete after all retries, cut at the last full sentence;
- `too_short`: shorter than the minimum length after all retries;
- `no_chinese`: step 1 never produced Chinese, so the text is not a real
  back-translation (check or exclude it);
- `not_english`: the kept output still contains Chinese (check or exclude it).

A second file, `<output name>_pairs.jsonl`, keeps each attacked text next to
its original, with the line of the original in the input file and the status:
`{"source_index": 0, "original": "...", "attacked": "...", "status": "ok"}`.
The evaluation uses it to check that every attacked text is matched with the
right original.

In [9]:
progress = read_progress()   # same reading as Section 7, ignores an incomplete line
assert len(progress) == len(texts), f"{len(texts) - len(progress)} texts not processed yet: run Section 7 again"

outputs = [progress[i]["text"] for i in range(len(texts))]
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_FILE.open("w", encoding="utf-8") as f:
    json.dump(outputs, f, ensure_ascii=False, indent=2)
print("Saved", len(outputs), "texts to", OUTPUT_FILE)

# Quick checks
print("Status:", dict(Counter(progress[i]["status"] for i in range(len(texts)))))
too_long = [i for i in range(len(texts)) if n_llada(outputs[i]) > length_bounds(texts[i])[1]]
assert not too_long, f"{len(too_long)} outputs exceed the length limit"
print("All outputs within the length limit: yes")
ratios = [n_llada(o) / n_llada(t) for o, t in zip(outputs, texts)]
print(f"Length ratio: min {min(ratios):.2f}, median {median(ratios):.2f}, max {max(ratios):.2f}")
print("\n--- Input ---\n", texts[0][:600])
print("\n--- Output ---\n", outputs[0][:600])


# Pairs file: each attacked text next to its original, for checking and reading
PAIRS_FILE = OUTPUT_FILE.with_name(OUTPUT_FILE.stem + "_pairs.jsonl")
with PAIRS_FILE.open("w", encoding="utf-8") as f:
    for k in range(len(texts)):
        f.write(json.dumps({"source_index": source_indices[k],    # line in the input file
                            "original": texts[k],
                            "attacked": outputs[k],
                            "status": progress[k]["status"]},
                           ensure_ascii=False) + "\n")
print("Saved pairs to", PAIRS_FILE)

Saved 600 texts to /content/drive/MyDrive/watermark_project/backtranslated_texts.json
Status: {'trimmed': 366, 'ok': 223, 'no_chinese': 8, 'too_short': 2, 'not_english': 1}
All outputs within the length limit: yes
Length ratio: min 0.21, median 0.94, max 1.10

--- Input ---
 The collapse of the Soviet Union is when one of the largest superpowers, like a big super-strong robot that had been fighting together for many years, suddenly falls apart all by itself. Sometimes, a robot might fall apart because they don't work together anymore, or maybe they've gotten too far away that it's hard for them to find each other.

In the case of the Soviet Union, it was like two big robots that had been united for a long time didn't agree anymore in how to fight. They started arguing with each other and then one of them started making their own rules that they didn't want to shar

--- Output ---
 The dissolution of the Soviet Union was like a giant robot that had stood side by side for years, suddenly


## 9. Data Validation

We load the 600 back-translated responses and verify
their alignment with the original watermarked dataset.

We also check the generation status of each response,
since some outputs were shortened or did not pass
the language validation checks.


In [1]:

import sys
from pathlib import Path
from collections import Counter

def find_project_root(start):
    for folder in [start, *start.parents]:
        if (folder / "results" / "Cleaned Data").is_dir():
            return folder
    raise FileNotFoundError("Project root not found")

ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "src"))

from detection import load_jsonl

wm = load_jsonl(
    ROOT / "results" / "Cleaned Data" / "watermarked_600.jsonl"
)

baseline = load_jsonl(
    ROOT / "results" / "attacks" / "baseline_original.jsonl"
)

pairs = load_jsonl(
    ROOT / "results" / "attacks" / "backtranslation"
    / "backtranslated_texts_pairs.jsonl"
)

assert len(wm) == len(baseline) == len(pairs) == 600

assert all(
    row["prompt"] == base["prompt"]
    for row, base in zip(wm, baseline)
)

assert all(
    pair["source_index"] == i
    and pair["original"] == wm[i]["completion"]
    and isinstance(pair["attacked"], str)
    and pair["attacked"].strip()
    for i, pair in enumerate(pairs)
)

print("Alignment verified: 600/600")
print("Generation statuses:", dict(Counter(p["status"] for p in pairs)))


Alignment verified: 600/600
Generation statuses: {'trimmed': 366, 'ok': 223, 'no_chinese': 8, 'too_short': 2, 'not_english': 1}



## 10. Watermark Detection

We apply the same local watermark detector used in the
previous experiments to all 600 back-translated responses.

We use the LLaDA tokenizer and a significance threshold
of 0.01. The detection results are saved together with
the original baseline scores and generation status.


In [2]:

from detection import detect, load_detector, save_jsonl

OUT_DIR = ROOT / "results" / "attacks" / "backtranslation"
OUT_PATH = OUT_DIR / "backtranslation_detection.jsonl"

if OUT_PATH.exists():
    print(f"Results already exist: {OUT_PATH}")
else:
    tokenizer, detector = load_detector(ROOT)
    results = []

    for i, pair in enumerate(pairs):
        score = detect(pair["attacked"], tokenizer, detector)
        base = baseline[i]

        results.append({
            "idx": i,
            "dataset": base["dataset"],
            "attack": "qwen_backtranslation",
            "generation_status": pair["status"],
            "prompt": wm[i]["prompt"],
            "original": pair["original"],
            "attacked": pair["attacked"],
            "original_p_value": base["p_value"],
            "original_detected": base["detected"],
            "original_green_fraction": base["green_fraction"],
            **score,
        })

        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}/600")

    save_jsonl(results, OUT_PATH)

    n_detected = sum(r["detected"] for r in results)
    print(f"\nBack-translations still detected: {n_detected}/600")


/Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processed 100/600
Processed 200/600
Processed 300/600
Processed 400/600
Processed 500/600
Processed 600/600
Saved 600 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/backtranslation/backtranslation_detection.jsonl

Back-translations still detected: 212/600



## 11. Results and Analysis

We calculate the Attack Success Rate (ASR) using the 583
responses originally detected as watermarked.

We also report the post-attack detection rate and compare
the results across generation status groups.

This distinction is important because many back-translated
responses were shortened to satisfy the length constraints.
We report 95% Wilson confidence intervals for the main rates.


In [3]:

from collections import defaultdict
from scipy.stats import binomtest

recs = load_jsonl(
    ROOT / "results" / "attacks" / "backtranslation"
    / "backtranslation_detection.jsonl"
)

assert len(recs) == 600

def wilson(k, n):
    ci = binomtest(k, n).proportion_ci(method="wilson")
    return f"[{ci.low:.1%}, {ci.high:.1%}]"

initial = [r for r in recs if r["original_detected"]]
n_removed = sum(not r["detected"] for r in initial)
n_detected = sum(r["detected"] for r in recs)

print("Overall results")
print(f"Detection rate: {n_detected}/600 = {n_detected/600:.1%} "
      f"{wilson(n_detected, 600)}")
print(f"ASR: {n_removed}/{len(initial)} = {n_removed/len(initial):.1%} "
      f"{wilson(n_removed, len(initial))}")

by_status = defaultdict(list)
for r in recs:
    by_status[r["generation_status"]].append(r)

print("\nResults by generation status")
for status, rows in sorted(by_status.items()):
    detected = sum(r["detected"] for r in rows)
    originally_detected = [r for r in rows if r["original_detected"]]
    removed = sum(not r["detected"] for r in originally_detected)
    print(
        f"{status:12s} | n={len(rows):3d} | "
        f"detection={detected/len(rows):6.1%} | "
        f"ASR={removed/len(originally_detected):6.1%}"
        if originally_detected else
        f"{status:12s} | n={len(rows):3d} | ASR=N/A"
    )


Overall results
Detection rate: 212/600 = 35.3% [31.6%, 39.2%]
ASR: 371/583 = 63.6% [59.7%, 67.4%]

Results by generation status
no_chinese   | n=  8 | detection= 37.5% | ASR= 62.5%
not_english  | n=  1 | detection=  0.0% | ASR=100.0%
ok           | n=223 | detection= 31.4% | ASR= 67.7%
too_short    | n=  2 | detection=  0.0% | ASR=100.0%
trimmed      | n=366 | detection= 38.0% | ASR= 60.8%



### Results Interpretation

Back-translation achieves an ASR of 63.6%, with
212 out of 600 responses still detected.

The 223 responses classified as "ok" reach an ASR
of 67.7%, compared with 60.8% for trimmed responses.

This suggests that the attack's effectiveness is not
only caused by text truncation.

However, the generation status does not guarantee that
the original meaning has been preserved.
